In [1]:
import pandas as pd
import re
import json


# ============================================================
# 1. CONFIGURATION: KEYWORDS USED FOR TRANSPARENT CLASSIFICATION
# ============================================================

HEADER_TERMS = {
    "components",
    "subcomponents",
    "themes",
    "parameters",
    "facets",
    "styles",
    "behaviors",
    "end points",
    "endpoints",
    "drivers",
}


MEDICAL_KEYWORDS = {
    "fsh",
    "hormone",
    "diagnosis",
    "sleep apnea",
    "metabolic",
    "immune",
    "basophil",
    "serotonin",
    "chromatin",
    "macronutrient",
    "chronic pain",
    "polygenic",
    "gene",
    "caffeine sensitivity",
    "parathyroid",
    "vision-check",
    "metabolic rate",
}


MENTAL_HEALTH_KEYWORDS = {
    "depression",
    "burnout",
    "hypomania",
    "hysteria",
    "psychoticism",
    "identity diffusion",
    "acculturative stress",
    "sense-of-coherence score",
}


SPIRITUAL_KEYWORDS = {
    "spiritual",
    "religious",
    "sufi",
    "quran",
    "bahá",
    "reiki",
    "astrology",
    "i ching",
    "gnostic",
    "jewish",
    "hindu",
    "mantra",
    "kabbalah",
    "islamic",
    "sikh",
    "buddhist",
    "scripture",
    "sacred text",
    "shabbat",
    "dhikr",
    "yoga",
    "pilgrimage",
    "holiness",
    "satya",
}


COMMUNICATION_KEYWORDS = {
    "talkativeness",
    "brevity",
    "language use",
    "sentence structure",
    "storytelling",
    "outspokenness",
    "frankness",
    "listening",
    "non-verbal communication",
    "spelling",
    "eye-contact",
}


SKILL_KEYWORDS = {
    "reasoning",
    "knowledge",
    "skills",
    "ability",
    "analysis",
    "calculations",
    "memory",
    "perception",
    "understanding",
    "comprehension",
    "filing",
    "troubleshooting",
    "computer skills",
    "robotics",
    "mathematical",
    "data analysis",
    "information retention",
}


PREFERENCE_KEYWORDS = {
    "preference",
    "preferred",
    "attitude",
    "orientation",
    "desire",
    "motivation",
    "needs",
    "affinity",
}


EXTERNAL_FACT_KEYWORDS = {
    "count",
    "hours",
    "years",
    "frequency",
    "km/week",
    "time/day",
    "months",
    "subscribers",
    "subscription",
    "passport",
    "nationality",
    "attendance",
    "contributions",
    "usage",
    "intake",
    "commute",
    "presence",
    "ratio",
    "stamps",
    "sessions",
    "per day",
    "per week",
}


# ============================================================
# 2. NORMALIZATION
# ============================================================

def normalize_facet(value):
    """
    Creates a normalized version of the facet while preserving
    the original value separately.
    """

    value = str(value).strip()

    # Remove leading numeric IDs such as:
    # "800. Sufi practice: ..."
    value = re.sub(r"^\d+\.\s*", "", value)

    # Replace special dashes
    value = value.replace("–", "-")
    value = value.replace("—", "-")

    # Split camel-case words:
    # SelfEsteem -> Self Esteem
    value = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", value)

    # Remove trailing punctuation used by category headers
    value = re.sub(r"[:;]+$", "", value)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()


# ============================================================
# 3. HEADER / MALFORMED ENTRY DETECTION
# ============================================================

def is_header_like(raw_facet, normalized_facet):

    raw_facet = raw_facet.strip()

    # Strong signal: category headings frequently end with ":"
    if raw_facet.endswith(":"):
        return True

    # Detect common category/header terminology
    for term in HEADER_TERMS:
        if term in normalized_facet:
            return True

    return False


# ============================================================
# 4. FACET TYPE CLASSIFICATION
# ============================================================

def classify_facet(normalized_facet, header_like):

    if header_like:
        return "header_or_malformed"

    if any(keyword in normalized_facet for keyword in MEDICAL_KEYWORDS):
        return "medical_or_biological"

    if any(keyword in normalized_facet for keyword in MENTAL_HEALTH_KEYWORDS):
        return "psychological_or_mental_health"

    if any(keyword in normalized_facet for keyword in SPIRITUAL_KEYWORDS):
        return "spiritual_or_religious_practice"

    if any(keyword in normalized_facet for keyword in COMMUNICATION_KEYWORDS):
        return "communication_or_interaction"

    if any(keyword in normalized_facet for keyword in SKILL_KEYWORDS):
        return "skill_or_ability"

    if any(keyword in normalized_facet for keyword in PREFERENCE_KEYWORDS):
        return "preference_or_attitude"

    if any(keyword in normalized_facet for keyword in EXTERNAL_FACT_KEYWORDS):
        return "biographical_or_external_fact"

    # Default category for traits and behaviours
    return "behavioral_or_personality"


# ============================================================
# 5. OBSERVABILITY
# ============================================================

def assign_observability(facet_type):

    if facet_type == "header_or_malformed":
        return "not_observable"

    if facet_type == "medical_or_biological":
        return "not_observable"

    if facet_type == "psychological_or_mental_health":
        return "not_observable"

    # These can only be evaluated when explicitly mentioned
    if facet_type == "biographical_or_external_fact":
        return "conditional"

    if facet_type == "spiritual_or_religious_practice":
        return "conditional"

    # For traits, skills and preferences:
    # score only if sufficient evidence exists
    return "conditional"


# ============================================================
# 6. SENSITIVITY
# ============================================================

def assign_sensitivity(facet_type, normalized_facet):

    high_sensitivity_terms = {
        "drug-use",
        "physical-violence",
        "kink",
        "nationality",
        "cultural identity",
        "childhood",
        "attachment",
        "diagnosis",
        "chronic pain",
    }

    if facet_type in {
        "medical_or_biological",
        "psychological_or_mental_health"
    }:
        return "high"

    if any(term in normalized_facet for term in high_sensitivity_terms):
        return "high"

    if facet_type in {
        "spiritual_or_religious_practice",
        "biographical_or_external_fact"
    }:
        return "medium"

    return "low"


# ============================================================
# 7. ABSTENTION REASONS
# ============================================================

def get_abstention_reason(facet_type, observability):

    if facet_type == "header_or_malformed":
        return (
            "Header-like or malformed catalogue entry; "
            "excluded from retrieval and scoring."
        )

    if facet_type == "medical_or_biological":
        return (
            "Requires medical, laboratory, genetic, diagnostic, "
            "or other external evidence."
        )

    if facet_type == "psychological_or_mental_health":
        return (
            "Should not be diagnosed or assigned from ordinary "
            "conversational text alone."
        )

    if observability == "conditional":
        return (
            "Score only when the conversation provides direct, "
            "specific evidence about the speaker."
        )

    return "Insufficient conversational evidence."


# ============================================================
# 8. SCORING DEFINITIONS
# ============================================================

def get_scoring_definition(facet_type, observability):

    if observability == "not_observable":
        return None

    if facet_type == "communication_or_interaction":
        return (
            "Use a 1-5 ordinal scale based only on directly observable "
            "conversational evidence. Abstain when the sample is too "
            "short or unrepresentative."
        )

    if facet_type == "skill_or_ability":
        return (
            "Use a 1-5 ordinal scale based on demonstrated or explicitly "
            "described ability. Do not infer competence from interest alone."
        )

    if facet_type == "preference_or_attitude":
        return (
            "Use a 1-5 ordinal scale based on explicit statements or "
            "repeated direct evidence. Abstain when preference is unclear."
        )

    return (
        "Use a 1-5 ordinal scale based on direct conversational evidence. "
        "Do not infer stable traits from isolated statements."
    )


# ============================================================
# 9. MAIN PREPROCESSING FUNCTION
# ============================================================

def preprocess_facets(df):

    processed_rows = []

    for index, value in enumerate(df["Facets"], start=1):

        raw_facet = "" if pd.isna(value) else str(value).strip()

        normalized_facet = normalize_facet(raw_facet)

        is_empty = normalized_facet == ""

        if is_empty:
            header_like = True
            facet_type = "header_or_malformed"
        else:
            header_like = is_header_like(
                raw_facet,
                normalized_facet
            )

            facet_type = classify_facet(
                normalized_facet,
                header_like
            )

        observability = assign_observability(
            facet_type
        )

        sensitivity = assign_sensitivity(
            facet_type,
            normalized_facet
        )

        retrieval_enabled = (
            not header_like
            and observability != "not_observable"
        )

        scoring_definition = get_scoring_definition(
            facet_type,
            observability
        )

        abstention_reason = get_abstention_reason(
            facet_type,
            observability
        )

        processed_rows.append({
            "facet_id": index,
            "raw_facet": raw_facet,
            "normalized_facet": normalized_facet,
            "facet_type": facet_type,
            "conversation_observable": observability,
            "sensitivity": sensitivity,
            "is_header_like": header_like,
            "retrieval_enabled": retrieval_enabled,
            "scoring_definition": scoring_definition,
            "abstention_reason": abstention_reason,
        })

    return pd.DataFrame(processed_rows)


# ============================================================
# 10. LOAD CSV
# ============================================================

INPUT_FILE = '/content/Facets Assignment.csv'

df = pd.read_csv(INPUT_FILE)

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 11. RUN PREPROCESSING
# ============================================================

enriched_df = preprocess_facets(df)


# ============================================================
# 12. SAVE OUTPUT
# ============================================================

OUTPUT_FILE = "enriched_facets.csv"

enriched_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 13. AUDIT SUMMARY
# ============================================================

audit_summary = {
    "total_rows": int(len(enriched_df)),
    "header_like_rows": int(
        enriched_df["is_header_like"].sum()
    ),
    "retrieval_enabled_rows": int(
        enriched_df["retrieval_enabled"].sum()
    ),
    "facet_type_counts": (
        enriched_df["facet_type"]
        .value_counts()
        .to_dict()
    ),
    "observability_counts": (
        enriched_df["conversation_observable"]
        .value_counts()
        .to_dict()
    ),
    "sensitivity_counts": (
        enriched_df["sensitivity"]
        .value_counts()
        .to_dict()
    ),
}


print("\n" + "=" * 70)
print("FACET AUDIT SUMMARY")
print("=" * 70)

print(json.dumps(
    audit_summary,
    indent=4
))


# ============================================================
# 14. SAVE AUDIT SUMMARY
# ============================================================

with open(
    "audit_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        audit_summary,
        file,
        indent=4
    )


# ============================================================
# 15. DISPLAY SAMPLE OUTPUT
# ============================================================

print("\nSample of enriched dataset:\n")

display(
    enriched_df.head(20)
)


print("\nFiles created:")
print("-", OUTPUT_FILE)
print("-", "audit_summary.json")

Original dataset shape: (399, 1)

Columns:
['Facets']

FACET AUDIT SUMMARY
{
    "total_rows": 399,
    "header_like_rows": 31,
    "retrieval_enabled_rows": 341,
    "facet_type_counts": {
        "behavioral_or_personality": 219,
        "spiritual_or_religious_practice": 35,
        "skill_or_ability": 32,
        "biographical_or_external_fact": 32,
        "header_or_malformed": 31,
        "medical_or_biological": 17,
        "preference_or_attitude": 12,
        "communication_or_interaction": 11,
        "psychological_or_mental_health": 10
    },
    "observability_counts": {
        "conditional": 341,
        "not_observable": 58
    },
    "sensitivity_counts": {
        "low": 298,
        "medium": 66,
        "high": 35
    }
}

Sample of enriched dataset:



,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason
0,1,Risktaking,risktaking,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
1,2,Naivety,naivety,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
2,3,Acidity,acidity,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
3,4,Democratic Leadership:,democratic leadership,header_or_malformed,not_observable,low,True,False,None,Header-like or malformed catalogue entry; excl...
4,5,Common-sense,common-sense,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
5,6,Hesitation,hesitation,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
6,7,Discontentment,discontentment,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
7,8,Overprotectiveness,overprotectiveness,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
8,9,Merriness,merriness,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
9,10,Emotionalism,emotionalism,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...



Files created:
- enriched_facets.csv
- audit_summary.json


In [2]:
import re


# ============================================================
# HELPER: SAFE KEYWORD / PHRASE MATCHING
# ============================================================

def contains_keyword(text, keyword):
    """
    Matches complete words or complete phrases.

    Examples:
    contains_keyword("macronutrient ratio", "ratio")
    -> True

    contains_keyword("desperation", "ratio")
    -> False
    """

    pattern = r"(?<!\w)" + re.escape(keyword) + r"(?!\w)"

    return bool(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    )


def contains_any_keyword(text, keywords):

    return any(
        contains_keyword(text, keyword)
        for keyword in keywords
    )


# ============================================================
# IMPROVED HEADER DETECTION
# ============================================================

def is_header_like(raw_facet, normalized_facet):

    raw_facet = raw_facet.strip()

    # A trailing colon is a strong signal of a catalogue heading.
    if raw_facet.endswith(":"):
        return True

    # Match complete words/phrases only.
    return contains_any_keyword(
        normalized_facet,
        HEADER_TERMS
    )


# ============================================================
# IMPROVED FACET CLASSIFICATION
# ============================================================

def classify_facet(normalized_facet, header_like):

    if header_like:
        return "header_or_malformed"

    if contains_any_keyword(
        normalized_facet,
        MEDICAL_KEYWORDS
    ):
        return "medical_or_biological"

    if contains_any_keyword(
        normalized_facet,
        MENTAL_HEALTH_KEYWORDS
    ):
        return "psychological_or_mental_health"

    if contains_any_keyword(
        normalized_facet,
        SPIRITUAL_KEYWORDS
    ):
        return "spiritual_or_religious_practice"

    if contains_any_keyword(
        normalized_facet,
        COMMUNICATION_KEYWORDS
    ):
        return "communication_or_interaction"

    if contains_any_keyword(
        normalized_facet,
        SKILL_KEYWORDS
    ):
        return "skill_or_ability"

    if contains_any_keyword(
        normalized_facet,
        PREFERENCE_KEYWORDS
    ):
        return "preference_or_attitude"

    if contains_any_keyword(
        normalized_facet,
        EXTERNAL_FACT_KEYWORDS
    ):
        return "biographical_or_external_fact"

    return "behavioral_or_personality"


# ============================================================
# RERUN PREPROCESSING
# ============================================================

enriched_df = preprocess_facets(df)

enriched_df.to_csv(
    "enriched_facets_v2.csv",
    index=False
)


# ============================================================
# CHECK THE PREVIOUS BUG
# ============================================================

display(
    enriched_df[
        enriched_df["raw_facet"]
        .str.contains(
            "Desperation",
            case=False,
            na=False
        )
    ]
)

,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason
33,34,Desperation,desperation,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...


In [3]:
# ============================================================
# FULL AUDIT INSPECTION
# ============================================================

print("=" * 70)
print("FACET TYPE COUNTS")
print("=" * 70)

print(
    enriched_df["facet_type"]
    .value_counts()
)


print("\n" + "=" * 70)
print("OBSERVABILITY COUNTS")
print("=" * 70)

print(
    enriched_df["conversation_observable"]
    .value_counts()
)


print("\n" + "=" * 70)
print("SENSITIVITY COUNTS")
print("=" * 70)

print(
    enriched_df["sensitivity"]
    .value_counts()
)


print("\n" + "=" * 70)
print("HEADER-LIKE ROWS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["is_header_like"] == True
    ][
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type"
        ]
    ]
)


print("\n" + "=" * 70)
print("MEDICAL / BIOLOGICAL FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "medical_or_biological"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "retrieval_enabled"
        ]
    ]
)


print("\n" + "=" * 70)
print("PSYCHOLOGICAL / MENTAL HEALTH FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "psychological_or_mental_health"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "retrieval_enabled"
        ]
    ]
)


print("\n" + "=" * 70)
print("SPIRITUAL / RELIGIOUS FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "spiritual_or_religious_practice"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "sensitivity"
        ]
    ]
)


print("\n" + "=" * 70)
print("ROWS EXCLUDED FROM RETRIEVAL")
print("=" * 70)

display(
    enriched_df[
        enriched_df["retrieval_enabled"] == False
    ][
        [
            "facet_id",
            "raw_facet",
            "facet_type",
            "abstention_reason"
        ]
    ]
)

FACET TYPE COUNTS
facet_type
behavioral_or_personality          228
spiritual_or_religious_practice     34
header_or_malformed                 31
biographical_or_external_fact       29
skill_or_ability                    28
medical_or_biological               15
preference_or_attitude              13
communication_or_interaction        11
psychological_or_mental_health      10
Name: count, dtype: int64

OBSERVABILITY COUNTS
conversation_observable
conditional       343
not_observable     56
Name: count, dtype: int64

SENSITIVITY COUNTS
sensitivity
low       304
medium     62
high       33
Name: count, dtype: int64

HEADER-LIKE ROWS


,facet_id,raw_facet,normalized_facet,facet_type
3,4,Democratic Leadership:,democratic leadership,header_or_malformed
21,22,HonestyHumility:,honesty humility,header_or_malformed
22,23,Relationship Building Themes:,relationship building themes,header_or_malformed
24,25,Numerical Reasoning Subcomponents:,numerical reasoning subcomponents,header_or_malformed
34,35,Affiliation Motivation:,affiliation motivation,header_or_malformed
48,49,Innovation and Creativity Components:,innovation and creativity components,header_or_malformed
49,50,Achievement Motivation:,achievement motivation,header_or_malformed
69,70,Work Styles,work styles,header_or_malformed
85,86,Leadership Potential:,leadership potential,header_or_malformed
90,91,Listening Skills Subcomponents:,listening skills subcomponents,header_or_malformed



MEDICAL / BIOLOGICAL FACETS


,facet_id,raw_facet,conversation_observable,retrieval_enabled
31,32,FSH level,not_observable,False
89,90,Sleep-disorder diagnosis,not_observable,False
124,125,Parathyroid-hormone level,not_observable,False
135,136,Chromatin-accessibility score,not_observable,False
149,150,Serotonin transporter availability,not_observable,False
162,163,Vision-check frequency,not_observable,False
164,165,Macronutrient ratio: fat,not_observable,False
229,230,Metabolic Rate (Low or High),not_observable,False
237,238,Macronutrient ratio: carbs,not_observable,False
244,245,Immune-response age,not_observable,False



PSYCHOLOGICAL / MENTAL HEALTH FACETS


,facet_id,raw_facet,conversation_observable,retrieval_enabled
57,58,Sense-of-coherence score,not_observable,False
60,61,Depression Symptoms,not_observable,False
83,84,Depression: Feelings of sadness and hopelessness,not_observable,False
102,103,Burnout Symptoms,not_observable,False
111,112,Psychoticism,not_observable,False
174,175,Depression (DEP),not_observable,False
176,177,Hypomania (Ma),not_observable,False
198,199,Psychological construct: Acculturative Stress ...,not_observable,False
201,202,Hysteria (Hy),not_observable,False
343,344,Psychological construct: Identity Diffusion level,not_observable,False



SPIRITUAL / RELIGIOUS FACETS


,facet_id,raw_facet,conversation_observable,sensitivity
37,38,Presence of Spiritual Pain,conditional,medium
42,43,Pilgrimage participation count,conditional,medium
103,104,Holiness,conditional,medium
134,135,800. Sufi practice: Sufi retreat attendance count,conditional,medium
136,137,644. Spiritual virtue: Humility practice index,conditional,medium
137,138,754. I Ching hexagram 36 resonance level,conditional,medium
142,143,596. Religious practice: Quran khatam cycles p...,conditional,medium
143,144,926. Bahá’í spiritual metric: Ridván festival ...,conditional,medium
156,157,607. Energy-healing practice: Reiki sessions /...,conditional,medium
157,158,692. Astrology: Rising sign is Scorpio,conditional,medium



ROWS EXCLUDED FROM RETRIEVAL


,facet_id,raw_facet,facet_type,abstention_reason
3,4,Democratic Leadership:,header_or_malformed,Header-like or malformed catalogue entry; excl...
21,22,HonestyHumility:,header_or_malformed,Header-like or malformed catalogue entry; excl...
22,23,Relationship Building Themes:,header_or_malformed,Header-like or malformed catalogue entry; excl...
24,25,Numerical Reasoning Subcomponents:,header_or_malformed,Header-like or malformed catalogue entry; excl...
31,32,FSH level,medical_or_biological,"Requires medical, laboratory, genetic, diagnos..."
34,35,Affiliation Motivation:,header_or_malformed,Header-like or malformed catalogue entry; excl...
48,49,Innovation and Creativity Components:,header_or_malformed,Header-like or malformed catalogue entry; excl...
49,50,Achievement Motivation:,header_or_malformed,Header-like or malformed catalogue entry; excl...
57,58,Sense-of-coherence score,psychological_or_mental_health,Should not be diagnosed or assigned from ordin...
60,61,Depression Symptoms,psychological_or_mental_health,Should not be diagnosed or assigned from ordin...


In [4]:
# ============================================================
# FINAL HEADER AUDIT
# Check header-like rows that do NOT end with a colon.
# These are the rows most likely to reveal false positives.
# ============================================================

non_colon_headers = enriched_df[
    (enriched_df["is_header_like"] == True)
    &
    (~enriched_df["raw_facet"].str.strip().str.endswith(":"))
]

print("HEADER-LIKE ROWS WITHOUT A TRAILING COLON:")
print("Count:", len(non_colon_headers))

display(
    non_colon_headers[
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type"
        ]
    ]
)

HEADER-LIKE ROWS WITHOUT A TRAILING COLON:
Count: 1


,facet_id,raw_facet,normalized_facet,facet_type
69,70,Work Styles,work styles,header_or_malformed
